In [1]:
include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")

evaluate (generic function with 1 method)

## 1. Prepare Environment

In [2]:
using RockSample

pomdp = RockSamplePOMDP(7, 8)
pomdp_name = "RS78"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# define convert_o function
function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::RockSamplePOMDP)
    vec = zeros(Float32, 3)
    vec[o] = 1.0f0
    return vec
end

# define process action function
function process_action(action::Int, action_space::UnitRange{Int})
    len = length(action_space)
    idx = action - first(action_space) + 1
    (idx < 1 || idx > len) && error("Action $action not in action space")
    onehot = zeros(Float32, len)
    onehot[idx] = 1.0f0
    return onehot
end

process_action (generic function with 1 method)

## 2. Prepare Parameters

In [3]:
state_dim = GetObsDim(env)
action_dim = length(action_space)
layer_size = 64
rnn_hidden_size = 64
gamma = discount(pomdp)
training_episodes = 10000
batch_size = 2048

RSState{8}([1, 1], Bool[1, 1, 0, 1, 0, 0, 1, 0])
Float32[0.0, 0.0, 1.0]


2048

## 3. Prepare PPO-RNN agent

In [4]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
using CUDA

agent = PPORNNAgent(action_space, action_dim, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.gpu) 

PPORNNAgent(Chain(LSTM(16 => 64), Dense(64 => 64, tanh), Dense(64 => 13)), Chain(LSTM(16 => 64), Dense(64 => 64, tanh), Dense(64 => 1)), (layers = ((cell = (Wi = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999)))),), (weight = Leaf(Ada

## 4. Train

In [5]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

CompositeException: TaskFailedException

    nested task error: Scalar indexing is disallowed.
    Invocation of getindex resulted in scalar indexing of a GPU array.
    This is typically caused by calling an iterating implementation of a method.
    Such implementations *do not* execute on the GPU, but very slowly on the CPU,
    and therefore should be avoided.
    
    If you want to allow scalar iteration, use `allowscalar` or `@allowscalar`
    to enable scalar iteration globally or for the operations in question.
    Stacktrace:
      [1] error(s::String)
        @ Base ./error.jl:35
      [2] errorscalar(op::String)
        @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:151
      [3] _assertscalar(op::String, behavior::GPUArraysCore.ScalarIndexing)
        @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:124
      [4] assertscalar(op::String)
        @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:1

## 5. Evaluation

In [ ]:
evaluate(env, agent; num_episodes=10000, max_steps=100) 

## (Todo) Save or plot the data from Train (rewards, losses, evals)